<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/junchao/%E2%80%9CProgram_B_ipynb%E2%80%9D(bozheng).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

# Setup

In [ ]:
import os
import sys
import itertools
import functools

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

from pathlib import Path

In [ ]:
pd.set_option('display.max_columns', 100)

## Data getting (if on Colab)

In [ ]:
import google.colab
google.colab.drive.mount("/content/drive")

Get ancillary data from Github

In [ ]:
!git clone https://github.com/marius-ne/CIE_ProjectB_Group13.git

## Load data

In [ ]:
os.chdir("CIE_ProjectB_Group13")

In [ ]:
os.getcwd()

**IMPORTANT:** You need to add a shortcut of the "programB" folder on GoogleDrive to your own "MyDrive" for this to work

In [ ]:
!ln -s /content/drive/MyDrive/programB data

In [ ]:
target_folder = "data/Data2"
current_folder = os.getcwd()

if Path(current_folder).name != target_folder:
    os.chdir(Path(current_folder) / Path(target_folder))
print(os.getcwd())


Convert data

In [ ]:
VARIABLES = {
      0: "TotalDeformation.csv",
      1: "DirectionalDeformation_X_axis.csv",
      2: "DirectionalDeformation_Y_axis.csv",
      3: "DirectionalDeformation_Z_axis.csv",
      4: "EquivalentStress.csv",
      5: "ShearStress_XY.csv",
      6: "ShearStress_XZ.csv",
      7: "ShearStress_YZ.csv",
  }
LOADS = {
    0: "Bigger_train",
    1: "Smaller_train",
}
SEASONS = {
    0: "Summer",
    1: "Winter",
}
HEALTHS = {
    0: "Perfect_structure",
    1: "ip_frst_Arc_defect_all_tracks_111",
    2: "ip_1and3track_3_arc_78910",
    3: "ip_first_track_3arc_78910",
    4: "Ip_1track_1_arc_345",
    5: "ip_3track_1_arc_678",
    6: "ip_2_arc_all_tracks_222",
}
TRAIN_CONFIGS = {
    0: "One_train_1st_track",
    1: "One_train_middle_track",
    2: "Two_trains_extreme_track_different_direction",
    3: "Two_trains_extreme_track_same_direction",
}
VARIABLE_NAMES = [var[:-4] for var in VARIABLES.values()]
NODE_NUMBERS = None

def combination_to_string(combination):
    train_config, load, season, health, variable = combination
    return f"{TRAIN_CONFIGS[train_config]}__{LOADS[load]}__{SEASONS[season]}__{HEALTHS[health]}__{VARIABLES[variable][:-4]}"

# Construct list of scenarios (combinations of train configs, loads, seasons, healths, variables)
#   Each scenario is a tuple of (train_config, load, season, health, variable), each encoded
#   as the corresponding key in the dictionaries above
combinations = itertools.product(
        TRAIN_CONFIGS.keys(),
        LOADS.keys(),
        SEASONS.keys(),
        HEALTHS.keys(),
        VARIABLES.keys()
    )
combinations = list(combinations)

# Group by variables, i.e. each group has all variables for one scenario
combinations_grouped_by_variable = [
    combinations[i:i + len(VARIABLES)] for i in range(0, len(combinations), len(VARIABLES))
]
# Group further by healths, i.e. each group has all healths for one scenario (train config, load, season)
combinations_grouped_by_health = [
    combinations_grouped_by_variable[i:i + len(HEALTHS)] for i in range(0, len(combinations_grouped_by_variable), len(HEALTHS))
]



In [ ]:
perfect_combinations = [c for c in combinations if c[3] == 6 ]
ipfirstarcdefectalltracks_combinations = [c for c in combinations if c[3] == 5 ]
perfect_combinations, ipfirstarcdefectalltracks_combinations

In [ ]:
combinations_grouped_by_health[0]

In [ ]:
def read_data_file(
    train_config: int = 0,
    load: int = 0,
    season: int = 0,
    health: int = 0,
    variable: int = 0,
):
  """Reads data according to format and provides the data-frame as-is, with
  the categorical variables added as columns."""

  # Construct filename from scenario according to the folder structure

  results_paths = ["Results", "Results1"]


  for results_path in results_paths:
      filename = Path()

      filename /= TRAIN_CONFIGS[train_config]
      filename /= LOADS[load]
      filename /= SEASONS[season]
      filename /= HEALTHS[health]
      filename /= results_path
      filename /= VARIABLES[variable]

      if filename.exists():
          break
  else:
      print(filename)
      raise FileNotFoundError(f"Data file not found for combination: {combination_to_string((train_config, load, season, health, variable))}")

  # Encode the scenario as categorical columns
  #   -> TODO: Is there a way of encoding that
  #   preserves information? E.g. like encoding the name of a city as its latitude
  df = pd.read_csv(filename)
  num_nodes = len(df)
  df["season"] = season*np.ones(num_nodes,dtype=np.uint8)
  df["health"] = health*np.ones(num_nodes,dtype=np.uint8)
  df["load"] = load*np.ones(num_nodes,dtype=np.uint8)
  df["train_config"] = train_config*np.ones(num_nodes,dtype=np.uint8)

  return df


def get_data():
  """
  Reads all data files from one health group and merges them
  into a single data-frame.
  TODO: Make it read all scenarios, not just one.

  Returns:
      pd.DataFrame: Merged data-frame with all variables as columns.
  """
  global NODE_NUMBERS

  # Select one scenario that has same season, load and trains and goes through
  #   all healths and variables
  scenario = combinations_grouped_by_health[0] # TBD

  # Merge data for all healths in the scenario
  dfs_healths = []
  for same_health_combinations in scenario:
    dfs_variables = []

    #  Merge data for all variables in the current health scenario
    for same_variable_combination in same_health_combinations:
      print("Processing combination:", combination_to_string(same_variable_combination))

      var_name = VARIABLE_NAMES[same_variable_combination[-1]]

      # Get data file for current combination
      df = read_data_file(*same_variable_combination)

      # Turn the variable column into a single one and add a new time column
      df_melted = df.melt(id_vars=["Node Number","season","load","health","train_config"],var_name="variable",value_name=var_name)
      df_melted["time"] = df_melted["variable"].str[-3:].astype(np.float64)
      df_melted.drop(columns=["variable"],inplace=True)

      # Get node numbers and ensure they're consistent
      if NODE_NUMBERS is None:
        NODE_NUMBERS = df_melted["Node Number"].unique()
      else:
        try:
          assert all(NODE_NUMBERS == df_melted["Node Number"].unique())
        except ValueError or AssertionError:
          print("WARNING: Node numbers differ between data files!")
          print("Previous node numbers:", NODE_NUMBERS)
          print("Current node numbers:", df_melted["Node Number"].unique())

      dfs_variables.append(df_melted)

    # Concatenating all variables into a single data frame
    # -> we do an OUTER join, meaning all keys are kept (A U B)
    #   this should be safe, node numbers and the other shared columns are kept
    shared_cols = ["Node Number","season","load","health","train_config","time"]
    df_vars = functools.reduce(lambda left,right: pd.merge(left,right,on=shared_cols,
                                              how='outer'), dfs_variables)
    # Check that data has been preserved
    for df in dfs_variables:
      for var_name in VARIABLE_NAMES:
        if var_name in df.columns:
          merged = pd.merge(df[shared_cols + [var_name]], df_vars[shared_cols + [var_name]],
                            on=shared_cols, how='inner')
          assert len(merged) == len(df)

    dfs_healths.append(df_vars)

  # Check that columns are the same
  assert all(all(df.columns == dfs_healths[0].columns) for df in dfs_healths)

  # Concatenate them together
  df = pd.concat(dfs_healths,ignore_index=True)

  return df

df = get_data()

In [ ]:
df

# Analyze node numbers

Go through all data files and check number of nodes. Create csv file that recaps these.

In [ ]:
nums = []
for ix, combination in enumerate(combinations):
    print(f"Processing combination {ix+1}/{len(combinations)}: {combination_to_string(combination)}")
    num_nodes = len(read_data_file(*combination)["Node Number"].unique())
    entry = {}
    entry["train_config"] = combination[0]
    entry["load"] = combination[1]
    entry["season"] = combination[2]
    entry["health"] = combination[3]
    entry["variable"] = combination[4]
    entry["num_nodes"] = num_nodes
    nums.append(entry)
nums

In [ ]:
os.listdir

Exemplary plot of node numbers throughout a number of scenarios.

In [ ]:
nn_df = pd.DataFrame(nums)
nn_df.plot(
    x="health", y="num_nodes", kind="bar",
    title=f"Num. nodes per health state"
)

In [ ]:
# Save recap to file
# nn_df.to_csv("node_numbers_Data2_overview.csv", index=False)

# Visualize bridge structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
os.listdir("/content/drive/MyDrive")[:50]


In [ ]:
import os

target = "nodeExport.txt"
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if target in files:
        print("找到了：", os.path.join(root, target))
        break


In [ ]:
# Read tab separated node export file
node_xyz = pd.read_csv(
    "/content/drive/MyDrive/programB/Data2/nodeExport.txt",
    sep="\t",
    engine="python"
)


In [ ]:
def select_df_subset(combination):
    """Selects a subset of the main data-frame according to the given combination.

    Args:
        combination (tuple): A tuple of (train_config, load, season, health, variable).
    """
    train_config, load, season, health, variable = combination
    var_name = VARIABLE_NAMES[variable]
    df_subset = df[
        (df["train_config"] == train_config) &
        (df["load"] == load) &
        (df["season"] == season) &
        (df["health"] == health)
    ][["Node Number", "time", var_name]]
    return df_subset

In [ ]:
from ipywidgets import interact, FloatSlider

# Colors for missing nodes
missing_color = 'r'
missing_nn = {}
colors = [
    missing_color if nn in missing_nn.values() else 'b'
    for nn in node_xyz["Node Number"]
]

cmap = plt.get_cmap('viridis')

# Colors for bridge loads
combination = (0, 0, 0, 3, 0)  # Example combination

def plot_bridge_loads_3d_slider(combination):
    """
    Plots bridge loads in 3D with a time slider.
    Args:
        combination (tuple): A tuple of (train_config, load, season, health, variable).
    """
    df_subset = select_df_subset(combination)
    time_points = np.sort(df_subset["time"].unique())
    variable_to_plot = VARIABLE_NAMES[combination[-1]]

    def plot_at_time(time_point_index):
        time_point = time_points[int(time_point_index)]
        timestamp_subset = df_subset[df_subset["time"] == time_point]
        node_loads = [
            timestamp_subset[timestamp_subset["Node Number"] == nn][variable_to_plot].values
            for nn in node_xyz["Node Number"]
        ]
        node_loads_flat = [nl[0] if len(nl) > 0 else np.nan for nl in node_loads]

        fig = plt.figure(figsize=(10,10))
        ax = fig.add_subplot(111, projection='3d')
        scatter = ax.scatter(
            node_xyz["X Location (m)"],
            node_xyz["Y Location (m)"],
            node_xyz["Z Location (m)"],
            c=node_loads_flat, marker='o', s=2,
            cmap="viridis"
        )
        fig.colorbar(scatter, shrink=0.5)
        ax.set_title(f"Bridge Loads at time {time_point}\nFor combination: {combination_to_string(combination)}")
        fig.savefig(f"../visualization/bridge_loads_3d_{time_point}.png")
        # fig.show()

    interact(
        plot_at_time,
        time_point_index=FloatSlider(
            min=0,
            max=len(time_points)-1,
            step=1,
            value=0,
            description='Time Index'
        )
    )

plot_bridge_loads_3d_slider(combination)

Create GIF from visualizations

In [ ]:
import numpy as np
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from pathlib import Path

def create_gif_bridge_loads(
    combination,
    viz_dir="/content/drive/MyDrive/programB (1)/CIE_ProjectB_Group13/Data2/CIE_ProjectB_Group13/visualization",
    gif_name="bridge_loads_animation.gif",
    duration=0.5,   # 秒/帧
    dpi=150
):
    viz_dir = Path(viz_dir)
    viz_dir.mkdir(parents=True, exist_ok=True)

    df_subset = select_df_subset(combination)
    time_points = np.sort(df_subset["time"].unique())
    if len(time_points) == 0:
        raise ValueError("df_subset 为空：这个 combination 在 df 里没有数据")

    var = VARIABLE_NAMES[combination[-1]]

    # （可选但推荐）固定色标范围，让不同时间的颜色可比
    vmin = df_subset[var].min(skipna=True)
    vmax = df_subset[var].max(skipna=True)

    frames = []

    for t in time_points:
        timestamp_subset = df_subset[df_subset["time"] == t]

        # 按 node_xyz 顺序取值，缺失填 NaN
        node_vals = []
        for nn in node_xyz["Node Number"]:
            arr = timestamp_subset.loc[timestamp_subset["Node Number"] == nn, var].values
            node_vals.append(arr[0] if len(arr) > 0 else np.nan)

        # 固定画布像素尺寸（关键：保证每帧shape一致）
        fig = plt.figure(figsize=(10, 10), dpi=dpi)
        canvas = FigureCanvas(fig)
        ax = fig.add_subplot(111, projection="3d")

        sc = ax.scatter(
            node_xyz["X Location (m)"],
            node_xyz["Y Location (m)"],
            node_xyz["Z Location (m)"],
            c=node_vals, s=2, cmap="viridis",
            vmin=vmin, vmax=vmax
        )
        fig.colorbar(sc, shrink=0.5)
        ax.set_title(f"Bridge Loads at time {t}\n{combination_to_string(combination)}")

        canvas.draw()
        frame = np.asarray(canvas.buffer_rgba())[:, :, :3]  # RGB
        frames.append(frame)

        plt.close(fig)

    gif_path = viz_dir / gif_name
    imageio.mimsave(gif_path, frames, duration=duration)
    print("GIF saved to:", gif_path)
    return gif_path

# 例子：用你之前的 combination
combination = (0, 0, 0, 3, 0)
gif_path = create_gif_bridge_loads(combination, duration=0.5)


# Remove NaN

In [ ]:
df.dropna(inplace=True)
df

# Data Inspection

In [ ]:
df.dtypes

In [ ]:
m = 5
df.iloc[:m*10].plot(subplots=True,figsize=(15,15))

In [ ]:
df_sorted = df.sort_values(by=["Node Number","health","time"], ascending=[True, True, True])
# Attention - make sure the indices are reset after sorting
df_sorted.reset_index(drop=True, inplace=True)
df_sorted

In [ ]:
df_sorted[["Node Number","health","time","EquivalentStress"]].iloc[:70].plot(subplots=True,figsize=(10,5))

Compare perfect and imperfect

In [ ]:
# Filter for health 0 and 1
df_comp = df_sorted[df_sorted["health"].isin([0, 1])]

df_pivot = df_comp.pivot_table(
    index=["Node Number","time","load","train_config","season"],
    columns="health",
    values=VARIABLE_NAMES
)
for variable in VARIABLE_NAMES:
    df_pivot[(variable, 'health_diff')] = df_pivot[(variable, 1)] - df_pivot[(variable, 0)]

df_diff = df_pivot[[ (var, 'health_diff') for var in VARIABLE_NAMES ]]
df_diff.columns = [var for var, _ in df_diff.columns]
df_diff.reset_index(inplace=True)
comp_diff_node_numbers = []
for var in VARIABLE_NAMES:
    for node_number in NODE_NUMBERS:
        if abs(df_diff[var][node_number]) > 0:
            comp_diff_node_numbers.append(node_number)
comp_diff_node_numbers = np.unique(comp_diff_node_numbers)
comp_diff_node_numbers

Isolate nodes with changing variables

In [ ]:
df_time_mean = df_sorted.groupby(by=["Node Number","health"]).mean().reset_index()
df_time_std = df_sorted.groupby(by=["Node Number","health"]).std().reset_index()

In [ ]:

varying_nodes = []
for node_number in NODE_NUMBERS:
    not_same = False
    for var in VARIABLE_NAMES:
        unique_means = df_time_mean[df_time_mean["Node Number"]==node_number][var].unique()
        unique_stds = df_time_std[df_time_std["Node Number"]==node_number][var].unique()
        if len(unique_means) > 1 or len(unique_stds) > 1:
            not_same = True
    if not_same:
        varying_nodes.append(int(node_number))
len(varying_nodes), len(NODE_NUMBERS)

In [ ]:
# Only keep nodes with varying variables
df_varying = df_sorted[df_sorted["Node Number"].isin(varying_nodes)]
df_varying

In [ ]:
df_varying.iloc[:140].plot(subplots=True,figsize=(10,20))

# Figure out bridge structure

NOTE: This is now obsolete because we have the node locations.

In [ ]:
len(NODE_NUMBERS)

In [ ]:
from ipywidgets import interact, IntSlider

def integer_factors(n):
    """Returns the list of integer factors of n."""
    factors = []
    for i in range(1, n + 1):
        if n % i == 0:
            factors.append(i)
    return factors
def plot_integer_widths(node_numbers_of_interest: list[int]):
    """Plots the bridge structure as images for all possible widths."""
    img_widths = integer_factors(len(NODE_NUMBERS))
    for w in img_widths:
        img = np.zeros((w,len(NODE_NUMBERS)//w))
        flat_img = img.flatten()
        for i,node_number in enumerate(NODE_NUMBERS):
            if node_number in node_numbers_of_interest:
                flat_img[i] = 1
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='gray', interpolation='nearest')
        plt.title(f'Node Variation Map (Width: {w})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
def plot_load_by_widths_interactive(df, timestep):
    img_widths = integer_factors(len(NODE_NUMBERS))
    times = df["time"].unique()
    if timestep not in times:
        raise ValueError(f"Timestep {timestep} not found in data. Available times: {times}")
    df_time = df[df["time"] == timestep]
    def plot_at_width(width_idx):
        w = img_widths[width_idx]
        img = np.zeros((w, len(NODE_NUMBERS)//w))
        flat_img = img.flatten()
        for i, node_number in enumerate(NODE_NUMBERS):
            load_value = df_time[df_time["Node Number"] == node_number]["TotalDeformation"].values
            assert len(load_value) <= 1, f"Multiple load values found for node {node_number} at time {timestep}"
            if len(load_value) == 1:
                flat_img[i] = load_value[0]
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        plt.title(f'Bridge Load Map at time {timestep} (Width: {w}), (Height: {len(NODE_NUMBERS)//w})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
    interact(plot_at_width, width_idx=IntSlider(min=0, max=len(img_widths)-1, step=1, value=0, description='Width Index'))

def plot_load_by_time_interactive(df, width):
    times = np.sort(df["time"].unique())
    def plot_at_time(time_idx):
        time = times[time_idx]
        df_time = df[df["time"] == time]
        img = np.zeros((width, len(NODE_NUMBERS)//width))
        flat_img = img.flatten()
        for i, node_number in enumerate(NODE_NUMBERS):
            load_value = df_time[df_time["Node Number"] == node_number]["TotalDeformation"].values
            assert len(load_value) <= 1, f"Multiple load values found for node {node_number} at time {time}"
            if len(load_value) == 1:
                flat_img[i] = load_value[0]
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        plt.title(f'Bridge Load Map at time {time} (Width: {width})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
    interact(plot_at_time, time_idx=IntSlider(min=0, max=len(times)-1, step=1, value=0, description='Time Index'))


plot_load_by_widths_interactive(df_sorted[df_sorted["health"]==0], timestep=0.1)
plot_load_by_time_interactive(df_sorted[df_sorted["health"]==0], width=34)

In [ ]:
2210/(42*3)

In [ ]:
plot_integer_widths(comp_diff_node_numbers)

# Training

In [ ]:
df_train = df_varying.copy()

In [ ]:
X_raw, y_raw = df_train.drop(columns=["health"]), df_train["health"]
X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
    X_raw, y_raw, test_size = 0.1, random_state = 0, shuffle=True,
)

In [ ]:
def standardize(X_train_raw, X_test_raw):
    scaler = sklearn.preprocessing.StandardScaler()

    scaler.fit(X_train_raw)
    X_train = scaler.transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    return X_train, X_test

X_train, X_test = standardize(X_train_raw, X_test_raw)

# Compare imperfect and imperfect

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def _assert_cols(df, cols, name="df"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} 缺少列: {missing}")

base_cols = ["Node Number","time","train_config","load","season","health"]
_assert_cols(df_sorted, base_cols, "df_sorted")
_assert_cols(df_sorted, VARIABLE_NAMES, "df_sorted")

df_sorted["Node Number"] = df_sorted["Node Number"].astype(np.int64)
df_sorted["health"] = df_sorted["health"].astype(np.int8)
df_sorted["season"] = df_sorted["season"].astype(np.int8)
df_sorted["load"] = df_sorted["load"].astype(np.int8)
df_sorted["train_config"] = df_sorted["train_config"].astype(np.int8)


In [ ]:
def compute_imperfect_node_scores(
    df: pd.DataFrame,
    variables: list,
    base_health: int = 0,
    imperfect_healths=(1,2,3,4,5,6),
    reducer: str = "rms",
    combine_vars: str = "sum",
    eps: float = 1e-12
):
    """
    For scenario(train_config,load,season):
      Perfect(health=0) vs each Imperfect(health=1..6)
      Compute the difference and normalise it, then aggregate by time to obtain the score for each node.
Return:
      final_scores: each Node Number a final score
    """
    scenario_cols = ["train_config","load","season"]
    key_cols = scenario_cols + ["Node Number","time"]

    results = []

    for scen_key, df_scen in df.groupby(scenario_cols, sort=False):
        df0 = df_scen[df_scen["health"] == base_health]
        if df0.empty:
            continue

        base = (
            df0[key_cols + list(variables)]
            .set_index(key_cols)
            .sort_index()
        )

        std_per_var = base[variables].std(axis=0)
        std_per_var = std_per_var.replace(0, np.nan)

        per_h_scores = {}

        for h in imperfect_healths:
            dfh = df_scen[df_scen["health"] == h]
            if dfh.empty:
                continue

            imp = (
                dfh[key_cols + list(variables)]
                .set_index(key_cols)
                .sort_index()
            )

            common_index = base.index.intersection(imp.index)
            if len(common_index) == 0:
                continue

            d = imp.loc[common_index, variables] - base.loc[common_index, variables]
            d_norm = d / (std_per_var + eps)

            if reducer == "rms":
                node_var = d_norm.groupby(level=3).apply(lambda x: np.sqrt((x**2).mean(axis=0)))
            elif reducer == "max":
                node_var = d_norm.abs().groupby(level=3).max()
            else:
                raise ValueError

            if combine_vars == "sum":
                node_score = node_var.sum(axis=1)
            elif combine_vars == "max":
                node_score = node_var.max(axis=1)
            else:
                raise ValueError

            per_h_scores[h] = node_score

        if not per_h_scores:
            continue

        df_scores = pd.DataFrame(per_h_scores)         # columns = healths
        df_scores["score"] = df_scores.max(axis=1)     # health for max
        df_scores = df_scores.reset_index().rename(columns={"index": "Node Number"})

        df_scores["train_config"], df_scores["load"], df_scores["season"] = scen_key
        results.append(df_scores)

    if not results:
        raise RuntimeError

    scores_by_scenario = pd.concat(results, ignore_index=True)

    final_scores = (
        scores_by_scenario.groupby("Node Number", as_index=False)["score"]
        .mean()
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

    return final_scores, scores_by_scenario



In [ ]:

final_scores, scores_by_scenario = compute_imperfect_node_scores(
    df_sorted,
    variables=VARIABLE_NAMES,
    reducer="rms",
    combine_vars="sum"
)

final_scores.head(20)

In [ ]:
# Top 1%
k = max(1, int(0.005 * len(final_scores)))
suspects = final_scores.head(k)["Node Number"].tolist()
k, suspects[:10]


In [ ]:
# Look at the "inflection point"
plt.figure()
plt.plot(final_scores["score"].values)
plt.title("Sorted node scores")
plt.xlabel("rank")
plt.ylabel("score")
plt.grid(True)
plt.show()


In [ ]:
# check node_xyz
_assert_cols(node_xyz, ["Node Number","X Location (m)","Y Location (m)","Z Location (m)"], "node_xyz")

viz = node_xyz.merge(final_scores, on="Node Number", how="left")
viz["score"] = viz["score"].fillna(0.0)

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(
    viz["X Location (m)"],
    viz["Y Location (m)"],
    viz["Z Location (m)"],
    c='gray',
    s=1,
    cmap="viridis"
)
fig.colorbar(sc, shrink=0.5, label="imperfect score")
ax.set_title("Imperfect-node score on 3D node locations")
plt.show()


In [ ]:
viz["is_suspect"] = viz["Node Number"].isin(suspects)

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(
    viz["X Location (m)"], viz["Y Location (m)"], viz["Z Location (m)"],
    c='gray', s=1, cmap="viridis", alpha=0.1
)
fig.colorbar(sc, shrink=0.5, label="imperfect score")

sdf = viz[viz["is_suspect"]]
ax.scatter(
    sdf["X Location (m)"], sdf["Y Location (m)"], sdf["Z Location (m)"],
    c="r", s=12, marker="o", alpha=0.9
)
ax.set_title(f"Top {k} suspect nodes highlighted")
plt.show()


Decision Tree

In [ ]:
# model = sklearn.tree.DecisionTreeClassifier()
# model.fit(X_train, y_train)

In [ ]:
# model.score(X_test, y_test), model.score(X_train, y_train)

Neural Network

In [ ]:
# model = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(1000,1000,1000),verbose=1)
# model.fit(X_train, y_train)

In [ ]:
# model.score(X_test, y_test), model.score(X_train, y_train)